# IMDB Movie Review Sentiment Analysis — Exploratory Data Analysis

**Course:** Statistical Machine Learning / Data Science Capstone  
**Dataset:** IMDB Movie Reviews (50 000 reviews, binary sentiment)  
**Splits:** Stratified 70 / 15 / 15 — train / validation / test (seed = 42)  
**Objective:** Understand text characteristics, class balance, and vocabulary before modelling  

---

In [ ]:
# ── Cell 1 ─ Imports ──────────────────────────────────────────────
from pathlib import Path
from collections import Counter
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print("⚠  wordcloud not installed — run `pip install wordcloud` to enable word-cloud cells.")

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
    print("⚠  transformers not installed — token-length cell will use word_count × 1.3 estimate.")

# ── Visual theme ──────────────────────────────────────────────────
plt.style.use('dark_background')
GOLD = '#e8c547'
POS_COLOR = '#2ecc71'
NEG_COLOR = '#e74c3c'
sns.set_palette([POS_COLOR, NEG_COLOR])

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'artifacts' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STOPWORDS = {
    'the','a','an','is','was','were','are','be','been','being','have','has','had',
    'do','does','did','will','would','could','should','may','might','shall','can',
    'need','dare','ought','used','to','of','in','for','on','with','at','by','from',
    'as','into','through','during','before','after','above','below','between','out',
    'off','over','under','again','further','then','once','here','there','when',
    'where','why','how','all','both','each','few','more','most','other','some',
    'such','no','nor','not','only','own','same','so','than','too','very','just',
    'but','and','or','if','because','until','while','about','up','it','its','this',
    'that','these','those','i','me','my','he','him','his','she','her','we','they',
    'them','their','you','your'
}

LABEL_MAP = {0: 'Negative', 1: 'Positive'}

print('Setup complete ✓')

In [ ]:
# ── Cell 2 ─ Load data splits ────────────────────────────────────
train = pd.read_csv(ROOT / 'data/raw/train.csv')
val   = pd.read_csv(ROOT / 'data/raw/val.csv')
test  = pd.read_csv(ROOT / 'data/raw/test.csv')

splits = {'Train': train, 'Validation': val, 'Test': test}

for name, df in splits.items():
    print(f'{name:12s}  shape = {str(df.shape):16s}  columns = {list(df.columns)}')

print('\n── Train head ──')
train.head(3)

In [ ]:
# ── Cell 3 ─ Class distribution bar charts ───────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (name, df) in zip(axes[:3], splits.items()):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar(
        [LABEL_MAP[i] for i in counts.index],
        counts.values,
        color=[NEG_COLOR, POS_COLOR],
        edgecolor='white', linewidth=0.5
    )
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}', ha='center', va='bottom', fontsize=11, color='white')
    ax.set_title(f'{name} split', fontsize=13, color=GOLD)
    ax.set_ylabel('Number of reviews')
    ax.set_xlabel('Sentiment')

# Stacked comparison
ax = axes[3]
split_names = list(splits.keys())
neg_counts = [df['label'].value_counts().get(0, 0) for df in splits.values()]
pos_counts = [df['label'].value_counts().get(1, 0) for df in splits.values()]
x = np.arange(len(split_names))
w = 0.55
ax.bar(x, neg_counts, w, label='Negative', color=NEG_COLOR, edgecolor='white', linewidth=0.5)
ax.bar(x, pos_counts, w, bottom=neg_counts, label='Positive', color=POS_COLOR, edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(split_names)
ax.set_title('Stacked comparison', fontsize=13, color=GOLD)
ax.set_ylabel('Number of reviews')
ax.legend()

fig.suptitle('Class Distribution Across Splits', fontsize=15, color=GOLD, y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'class_balance.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {RESULTS_DIR / "class_balance.png"}')

In [ ]:
# ── Cell 4 ─ Split size summary table ────────────────────────────
summary_rows = []
for name, df in splits.items():
    vc = df['label'].value_counts()
    summary_rows.append({
        'Split': name,
        'Total': len(df),
        'Positive': vc.get(1, 0),
        'Negative': vc.get(0, 0),
        'Pos %': f"{vc.get(1, 0) / len(df) * 100:.1f}%",
        'Neg %': f"{vc.get(0, 0) / len(df) * 100:.1f}%",
    })

split_summary = pd.DataFrame(summary_rows)
split_summary.style.set_caption('Dataset Split Summary')

In [ ]:
# ── Cell 5 ─ Text length histogram (characters) ─────────────────
train['char_len'] = train['text'].astype(str).str.len()
train['sentiment'] = train['label'].map(LABEL_MAP)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, color, name) in zip(axes, [(1, POS_COLOR, 'Positive'), (0, NEG_COLOR, 'Negative')]):
    subset = train.loc[train['label'] == label, 'char_len']
    ax.hist(subset, bins=60, color=color, alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.axvline(subset.median(), color=GOLD, linestyle='--', linewidth=1.5, label=f'Median = {subset.median():.0f}')
    ax.set_title(f'{name} reviews', fontsize=13, color=GOLD)
    ax.set_xlabel('Character length')
    ax.set_ylabel('Frequency')
    ax.legend()

fig.suptitle('Review Length Distribution (Characters)', fontsize=15, color=GOLD, y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'text_length_dist.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {RESULTS_DIR / "text_length_dist.png"}')

In [ ]:
# ── Cell 6 ─ Text length statistics table ────────────────────────
stats = (
    train.groupby('sentiment')['char_len']
    .agg(['mean', 'median', 'std', 'min', 'max', 'count'])
    .round(1)
)
stats.columns = ['Mean', 'Median', 'Std Dev', 'Min', 'Max', 'Count']
stats.style.set_caption('Character-Length Statistics by Sentiment').format(precision=1)

In [ ]:
# ── Cell 7 ─ Word count distribution ─────────────────────────────
train['word_count'] = train['text'].astype(str).str.split().str.len()

fig, ax = plt.subplots(figsize=(12, 5))
for label, color, name in [(1, POS_COLOR, 'Positive'), (0, NEG_COLOR, 'Negative')]:
    subset = train.loc[train['label'] == label, 'word_count']
    ax.hist(subset, bins=60, color=color, alpha=0.6, edgecolor='white', linewidth=0.3, label=name)

ax.axvline(train['word_count'].median(), color=GOLD, linestyle='--', linewidth=1.5,
           label=f'Overall median = {train["word_count"].median():.0f}')
ax.set_title('Word Count Distribution by Sentiment', fontsize=14, color=GOLD)
ax.set_xlabel('Word count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8 ─ Token length (DistilBERT tokenizer) ─────────────────
TRUNCATION_THRESHOLD = 256

if HAS_TRANSFORMERS:
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    sample = train['text'].astype(str).tolist()
    token_lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in sample]
    method_note = 'DistilBERT tokenizer'
else:
    token_lengths = (train['word_count'] * 1.3).astype(int).tolist()
    method_note = 'Estimated (word_count × 1.3)'

train['token_len'] = token_lengths
exceed = sum(1 for t in token_lengths if t > TRUNCATION_THRESHOLD)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(token_lengths, bins=80, color=GOLD, alpha=0.85, edgecolor='white', linewidth=0.3)
ax.axvline(TRUNCATION_THRESHOLD, color=NEG_COLOR, linestyle='--', linewidth=2,
           label=f'Truncation @ {TRUNCATION_THRESHOLD} tokens')
ax.set_title(f'Token Length Distribution — {method_note}', fontsize=14, color=GOLD)
ax.set_xlabel('Number of tokens')
ax.set_ylabel('Frequency')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

pct = exceed / len(token_lengths) * 100
print(f'Reviews exceeding {TRUNCATION_THRESHOLD} tokens: {exceed:,} / {len(token_lengths):,} ({pct:.1f}%)')
print(f'Tokenisation method: {method_note}')

In [ ]:
# ── Cell 9 ─ Top-30 most frequent words ──────────────────────────
def tokenize_simple(texts):
    """Lowercase split, strip punctuation, remove stopwords."""
    import re
    for t in texts:
        for w in re.findall(r"[a-z]+", str(t).lower()):
            if w not in STOPWORDS and len(w) > 1:
                yield w

word_freq = Counter(tokenize_simple(train['text']))
top30 = word_freq.most_common(30)

words, counts = zip(*top30)
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(len(words)), counts, color=GOLD, edgecolor='white', linewidth=0.3)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_xlabel('Frequency')
ax.set_title('Top-30 Most Frequent Words (excluding stopwords)', fontsize=14, color=GOLD)
for bar, c in zip(bars, counts):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{c:,}', va='center', fontsize=9, color='white')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10 ─ Top-20 bigrams per class ───────────────────────────
import re

def get_bigrams(texts):
    bigram_counter = Counter()
    for t in texts:
        words = [w for w in re.findall(r"[a-z]+", str(t).lower())
                 if w not in STOPWORDS and len(w) > 1]
        bigram_counter.update(zip(words[:-1], words[1:]))
    return bigram_counter

pos_bigrams = get_bigrams(train.loc[train['label'] == 1, 'text']).most_common(20)
neg_bigrams = get_bigrams(train.loc[train['label'] == 0, 'text']).most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, bigrams, color, title in [
    (axes[0], pos_bigrams, POS_COLOR, 'Positive reviews'),
    (axes[1], neg_bigrams, NEG_COLOR, 'Negative reviews'),
]:
    labels = [f'{a} {b}' for (a, b), _ in bigrams]
    vals   = [c for _, c in bigrams]
    ax.barh(range(len(labels)), vals, color=color, edgecolor='white', linewidth=0.3)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(f'Top-20 Bigrams — {title}', fontsize=13, color=GOLD)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11 ─ Word cloud — Positive reviews ─────────────────────
if HAS_WORDCLOUD:
    pos_text = ' '.join(train.loc[train['label'] == 1, 'text'].astype(str))
    wc_pos = WordCloud(
        width=1200, height=600, background_color='black',
        colormap='Greens', max_words=200, stopwords=STOPWORDS,
        collocations=False
    ).generate(pos_text)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc_pos, interpolation='bilinear')
    ax.set_title('Word Cloud — Positive Reviews', fontsize=15, color=POS_COLOR)
    ax.axis('off')
    fig.savefig(RESULTS_DIR / 'wordcloud_pos.png', dpi=150, bbox_inches='tight', facecolor='black')
    plt.show()
    print(f'Saved → {RESULTS_DIR / "wordcloud_pos.png"}')
else:
    print('Skipped — install wordcloud: pip install wordcloud')

In [ ]:
# ── Cell 12 ─ Word cloud — Negative reviews ─────────────────────
if HAS_WORDCLOUD:
    neg_text = ' '.join(train.loc[train['label'] == 0, 'text'].astype(str))
    wc_neg = WordCloud(
        width=1200, height=600, background_color='black',
        colormap='Reds', max_words=200, stopwords=STOPWORDS,
        collocations=False
    ).generate(neg_text)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc_neg, interpolation='bilinear')
    ax.set_title('Word Cloud — Negative Reviews', fontsize=15, color=NEG_COLOR)
    ax.axis('off')
    fig.savefig(RESULTS_DIR / 'wordcloud_neg.png', dpi=150, bbox_inches='tight', facecolor='black')
    plt.show()
    print(f'Saved → {RESULTS_DIR / "wordcloud_neg.png"}')
else:
    print('Skipped — install wordcloud: pip install wordcloud')

In [ ]:
# ── Cell 13 ─ Vocabulary overlap (Jaccard similarity) ────────────
pos_vocab = set(tokenize_simple(train.loc[train['label'] == 1, 'text']))
neg_vocab = set(tokenize_simple(train.loc[train['label'] == 0, 'text']))

intersection = pos_vocab & neg_vocab
union        = pos_vocab | neg_vocab
jaccard      = len(intersection) / len(union)

pos_only = pos_vocab - neg_vocab
neg_only = neg_vocab - pos_vocab

print(f'Positive vocabulary size : {len(pos_vocab):,}')
print(f'Negative vocabulary size : {len(neg_vocab):,}')
print(f'Intersection             : {len(intersection):,}')
print(f'Union                    : {len(union):,}')
print(f'Jaccard similarity       : {jaccard:.4f}')
print(f'\nPositive-only words (sample): {sorted(list(pos_only))[:20]}')
print(f'Negative-only words (sample): {sorted(list(neg_only))[:20]}')

# Venn-style bar
fig, ax = plt.subplots(figsize=(10, 4))
categories = ['Positive only', 'Shared', 'Negative only']
values = [len(pos_only), len(intersection), len(neg_only)]
colors = [POS_COLOR, GOLD, NEG_COLOR]
bars = ax.bar(categories, values, color=colors, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{v:,}', ha='center', va='bottom', fontsize=11, color='white')
ax.set_title(f'Vocabulary Overlap — Jaccard = {jaccard:.4f}', fontsize=14, color=GOLD)
ax.set_ylabel('Unique words')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14 ─ Review length vs sentiment — box plot ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length
sns.boxplot(
    data=train, x='sentiment', y='char_len', order=['Positive', 'Negative'],
    palette=[POS_COLOR, NEG_COLOR], ax=axes[0],
    flierprops=dict(marker='o', markersize=2, alpha=0.3)
)
axes[0].set_title('Character Length by Sentiment', fontsize=13, color=GOLD)
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Character length')

# Word count
sns.boxplot(
    data=train, x='sentiment', y='word_count', order=['Positive', 'Negative'],
    palette=[POS_COLOR, NEG_COLOR], ax=axes[1],
    flierprops=dict(marker='o', markersize=2, alpha=0.3)
)
axes[1].set_title('Word Count by Sentiment', fontsize=13, color=GOLD)
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word count')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 15 ─ Sample reviews — shortest and longest ──────────────
print('═' * 80)
print('3 SHORTEST REVIEWS')
print('═' * 80)
shortest = train.nsmallest(3, 'char_len')[['text', 'label', 'char_len']]
for _, row in shortest.iterrows():
    print(f"\n[{LABEL_MAP[row['label']]}]  ({row['char_len']} chars)")
    print(f"  \"{row['text'][:300]}\"")

print('\n' + '═' * 80)
print('3 LONGEST REVIEWS')
print('═' * 80)
longest = train.nlargest(3, 'char_len')[['text', 'label', 'char_len']]
for _, row in longest.iterrows():
    print(f"\n[{LABEL_MAP[row['label']]}]  ({row['char_len']:,} chars)")
    print(f"  \"{row['text'][:300]}…\"")

In [ ]:
# ── Cell 16 ─ Missing values / data quality check ────────────────
print('── Missing values ──')
for name, df in splits.items():
    nulls = df.isnull().sum()
    print(f'\n{name}:')
    print(nulls.to_string())

print('\n── Duplicate reviews (train) ──')
n_dup = train['text'].duplicated().sum()
print(f'Exact duplicate texts: {n_dup} ({n_dup / len(train) * 100:.2f}%)')

print('\n── Label value counts (sanity check) ──')
for name, df in splits.items():
    unique = sorted(df['label'].unique())
    print(f'{name:12s}  unique labels = {unique}')

print('\n── Empty / whitespace-only reviews (train) ──')
empty = train['text'].astype(str).str.strip().eq('').sum()
print(f'Empty reviews: {empty}')

print('\nData quality check ✓')

---

## Summary & Key Findings

| Aspect | Observation |
|--------|-------------|
| **Class balance** | The dataset is well-balanced (≈ 50 / 50) across all three splits, confirming the stratified split was applied correctly. |
| **Review length** | Reviews have a wide range of lengths (tens to thousands of characters). The median character length is similar for both classes, suggesting length alone is not a strong discriminator. |
| **Tokenisation** | A non-trivial fraction of reviews exceed the 256-token truncation threshold used by our DistilBERT model. This means some information loss is expected for long reviews. |
| **Vocabulary** | High Jaccard similarity between positive and negative vocabularies — most words appear in both classes. Sentiment differences are subtle and context-dependent, justifying the use of contextual embeddings (e.g., DistilBERT) over bag-of-words. |
| **Common words** | Words like *movie*, *film*, *one*, *good*, *great*, *bad* dominate. Bigram analysis reveals class-specific phrases (e.g., *waste time* for negative, *well worth* for positive). |
| **Data quality** | No missing values detected. Minimal duplicate reviews. All labels are binary (0 / 1). |

### Implications for Modelling

1. **Balanced classes** → standard cross-entropy loss is appropriate; no class weighting needed.  
2. **Long-tail length distribution** → truncation at 256 tokens trades off coverage for speed; consider 512 if compute allows.  
3. **High vocabulary overlap** → simple bag-of-words will struggle; contextual models (DistilBERT) are well-motivated.  
4. **Clean data** → minimal preprocessing needed beyond standard tokeniser handling.

---
*EDA complete. Next: `02_error_analysis.ipynb` (after `make error-analysis`) · `03_model_comparison.ipynb` (after `make capstone`).*